In [1]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from dotenv import load_dotenv
import os
import sys

# Set up path for src/
sys.path.append(os.path.abspath(".."))
load_dotenv()

from src.fetch_data import get_stock_candles
from src.news import get_company_news
from src.sentiment import analyze_sentiment
from src.preprocess import merge_price_sentiment



ImportError: cannot import name 'analyze_sentiment' from partially initialized module 'src.sentiment' (most likely due to a circular import) (/Users/tarangsonkusare/Documents/Person projects/Stock Market Predictor/src/sentiment.py)

In [ ]:
symbol = "AAPL"
today = datetime.today()
start_date = (today - timedelta(days=365)).strftime('%Y-%m-%d')
news_start = (today - timedelta(days=7)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')


In [ ]:
# 📉 Fetch historical price data
price_df = get_stock_candles(symbol, start_date=start_date, end_date=end_date)
price_df.head()



# 📰 Fetch news data
news_df = get_company_news(symbol, symbol, news_start, end_date)
sentiment_df = analyze_sentiment(news_df)
sentiment_df.head()


/Users/tarangsonkusare/Documents/Person projects/Stock Market Predictor/src/fetch_data.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval=interval)
[*********************100%***********************]  1 of 1 completed


,date,avg_sentiment
0,2025-06-22,-0.359200
1,2025-06-23,0.171100
2,2025-06-24,0.008586
3,2025-06-25,0.090830
4,2025-06-26,0.022482


In [ ]:
# Combine price and sentiment into one DataFrame
merged_df = merge_price_sentiment(price_df, sentiment_df)

# Create new features:
# - return: daily price % change
# - sentiment_lag1: previous day's average sentiment
# - target: next day's closing price (for prediction)
merged_df['return'] = merged_df['close'].pct_change()
merged_df['sentiment_lag1'] = merged_df['avg_sentiment'].shift(1)
merged_df['target'] = merged_df['close'].shift(-1)

# Drop rows with missing values
merged_df = merged_df.dropna()

# Show the first few processed rows
merged_df.head()


KeyError: 'timestamp'

In [ ]:
# Create a dual-axis plot to show both stock price and sentiment over time
fig, ax1 = plt.subplots(figsize=(14,6))

# Price chart
ax1.plot(merged_df['timestamp'], merged_df['close'], label='Close Price', color='blue')
ax1.set_ylabel('Price', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Sentiment chart
ax2 = ax1.twinx()
ax2.plot(merged_df['timestamp'], merged_df['avg_sentiment'], label='Sentiment', color='red', alpha=0.3)
ax2.set_ylabel('Sentiment', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Final formatting
plt.title(f"{symbol} Price & Sentiment Over Time")
plt.show()


MergeError: Not allowed to merge between different levels. (2 levels on the left, 1 on the right)

In [ ]:
# Select features for prediction
X = merged_df[['return', 'sentiment_lag1']]
y = merged_df['target']  # the actual next-day closing price

# Initialize and train the model
model = LinearRegression()
model.fit(X, y)

# Predict on the same data (just for evaluation)
y_pred = model.predict(X)

# Calculate error (RMSE)
rmse = mean_squared_error(y, y_pred, squared=False)
print(f"RMSE: {rmse:.4f}")


In [ ]:
# Compare predicted and actual prices visually
plt.figure(figsize=(12,5))
plt.plot(y.values, label='Actual Price', linewidth=2)
plt.plot(y_pred, label='Predicted Price', linestyle='--')
plt.title(f"{symbol} - Actual vs Predicted Next-Day Close")
plt.legend()
plt.show()


In [ ]:
# Get the latest row of features
latest = X.iloc[-1:]

# Predict next day's closing price
predicted_price = model.predict(latest)[0]
current_price = merged_df['close'].iloc[-1]

# Display the prediction
print(f"Current Price: ${current_price:.2f}")
print(f"Predicted Next Price: ${predicted_price:.2f}")
print(f"Expected Return: {(predicted_price - current_price) / current_price:.2%}")
